Construction of 2D COFs from User-Defined Nodes and Linkers Based on hcb or sql Topologies in perfect AA stacking with user-defined Interlayer Distance

IMPORT PACKAGES

In [48]:
import pormake as pm
import os
import tempfile
from pymatgen.core import Structure

DEFINE FUNCTIONS

In [49]:
def update_cgd_file(new_value, topo):
    # Define the paths
    pickle_path = os.path.expanduser(f"~/tools/anaconda3/envs/pormake/lib/python3.9/site-packages/pormake/database/topologies/{topo}_modified.pickle")
    cgd_path = os.path.expanduser(f"~/tools/anaconda3/envs/pormake/lib/python3.9/site-packages/pormake/database/topologies/{topo}_modified.cgd")

    # Step 1: Delete the .pickle file if it exists
    if os.path.exists(pickle_path):
        os.remove(pickle_path)
        print(f"Deleted file: {pickle_path}")
    else:
        print(f"File not found: {pickle_path}")

    # Step 2: Open and modify the .cgd file
    with open(cgd_path, 'r') as file:
        lines = file.readlines()

    line_parts = lines[3].split()  # Split line 4 into parts
    line_parts[3] = f"{float(new_value):.5f}"  # Replace the 4th value with new_value
    lines[3] = "  ".join(line_parts) + "\n"  # Reconstruct the line
    with open(cgd_path, 'w') as file:
        file.writelines(lines)


In [50]:
def extract_cell_lengths(cif_file):
    structure = Structure.from_file(cif_file)
    a = structure.lattice.a
    b = structure.lattice.b
    c = structure.lattice.c
    print(f"Extracted cell lengths: a = {a}, b = {b}, c = {c}")
    
    return a, b, c

In [51]:
def calculate_gamma(a, topo):
    c = 15 # rough guess value for ILD, not very important, will be changed later anyways with the script change_2d.py, just has to be large enough that layers do not overlap
    alpha = 1.73205 if topo == "hcb" else 1.0

    gamma = (alpha * c) / a # Scaling Value in .cgd file for c
    
    return gamma

DEFINE TOPOLOGY

In [52]:
# choose between hcb and sql
# hcb is based on 3D-bnn topology, sql is based on 3D-pcu topology (for both connecitvity in z-direction removed)
database = pm.Database()
topo = "sql"
topo_initial = database.get_topo(f"{topo}_initial")
topo_initial.describe()
# topo_initial.view()
topo_modified = database.get_topo(f"{topo}_modified")

Topology sql
Spacegroup: P4/mmm
-------------------------------------------------------------------------------
# of slots: 3 (1 nodes, 2 edges)
# of node types: 1
# of edge types: 1

-------------------------------------------------------------------------------
Node type information
-------------------------------------------------------------------------------
Node type: 0, CN: 4
  slot indices: 0

-------------------------------------------------------------------------------
Edge type information (adjacent node types) 
-------------------------------------------------------------------------------
Edge type: (0, 0)
  slot indices: 1, 2


DEFINE NODES AND LINKERS

In [53]:
# this example script is done for single node, single linker, single topology, for multiple linkers/nodes at once use loops and store them in lists

# NODE
# All xyz-node-files are stored in the input folder "nodes" in the same directory as this script
# define your node as a building block
nodename = "eva-node_2"
Node = pm.BuildingBlock(f"nodes/{nodename}.xyz")
# Node.view()

# LINKER
# All xyz-linker-files are stored in the input folder "linker" in the same directory as this script
linkername = "eva-imidazol"
Linker = pm.BuildingBlock(f"linker/{linkername}.xyz")
# Linker.view()

# Define a subfolder for the output files
output_folder = "cofs_test"
os.makedirs(output_folder, exist_ok=True)

BUILD COF

In [54]:
# Standart Setup, see orginal PORMAKE example for desription
builder = pm.Builder()
edgetype_raw = topo_initial.edge_types
edgetype_filtered = []
unique_pairs = set()
filtered_edge = edgetype_raw[(edgetype_raw != -1).any(axis=1)]
for pair in filtered_edge:
    if tuple(pair) not in unique_pairs:
        unique_pairs.add(tuple(pair))
        edgetype_filtered.append(tuple(pair))
node_bbs = {
    0: Node
}
edge_bbs = {pair: Linker for pair in edgetype_filtered}

# Build the COFs with initial gamma value of 1.0 to get the correct cell lengths a, b
cof = builder.build_by_type(topology=topo_initial, node_bbs=node_bbs, edge_bbs=edge_bbs)

# Option 3 (tempfile): write temporary CIF, read lengths, then delete
with tempfile.NamedTemporaryFile(suffix=".cif", delete=False) as tmp:
    tmp_path = tmp.name
cof.write_cif(tmp_path)
a, b, c = extract_cell_lengths(tmp_path)
os.remove(tmp_path)

# Now get the correct gamma value to update the cdg file to obtain the correct c
gamma = calculate_gamma(a, topo)
update_cgd_file(gamma, topo)

# Now build the real COF
topo_modified = database.get_topo(f"{topo}_modified")
cof = builder.build_by_type(topology=topo_modified, node_bbs=node_bbs, edge_bbs=edge_bbs)
output_filename = os.path.join(output_folder, f"{nodename}_{linkername}.cif")
cof.write_cif(output_filename)

>>> == Min RMSD of (node type: 0, node bb: eva-node_2): 7.88E-02
>>> Pre-location at node slot 0, (node type: 0, node bb: eva-node_2), RMSD: 7.88E-02
>>> Topology optimization starts.
>>> Pre-location at node slot 0, (node type: 0, node bb: eva-node_2), RMSD: 7.88E-02
>>> Topology optimization starts.
>>> MESSAGE: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
>>> SUCCESS: True
>>> ITER: 7
>>> OBJ: 0.000
>>> Location at node slot 0, (node type: 0, node bb: eva-node_2), RMSD: 4.85E-03
>>> Start placing edges.
/Users/gregorlauter/tools/anaconda3/envs/pormake/lib/python3.9/site-packages/pormake/locator.py:20: UserWarning: Optimal rotation is not uniquely or poorly defined for the given sets of vectors.
  U, rmsd = scipy.spatial.transform.Rotation.align_vectors(p, q)
>>> Start finding bonds in generated framework.
>>> Start finding bonds in building blocks.
>>> Start finding bonds between building blocks.
>>> Start making Framework instance.
>>> Construction of framework done.
>>> MESSAG

Extracted cell lengths: a = 25.398, b = 25.23, c = 19.25
Deleted file: /Users/gregorlauter/tools/anaconda3/envs/pormake/lib/python3.9/site-packages/pormake/database/topologies/sql_modified.pickle


>>> == Min RMSD of (node type: 0, node bb: eva-node_2): 7.88E-02
>>> Pre-location at node slot 0, (node type: 0, node bb: eva-node_2), RMSD: 7.88E-02
>>> Topology optimization starts.
>>> Pre-location at node slot 0, (node type: 0, node bb: eva-node_2), RMSD: 7.88E-02
>>> Topology optimization starts.
>>> MESSAGE: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
>>> SUCCESS: True
>>> ITER: 7
>>> OBJ: 0.000
>>> Location at node slot 0, (node type: 0, node bb: eva-node_2), RMSD: 4.85E-03
>>> MESSAGE: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
>>> SUCCESS: True
>>> ITER: 7
>>> OBJ: 0.000
>>> Location at node slot 0, (node type: 0, node bb: eva-node_2), RMSD: 4.85E-03
>>> Start placing edges.
/Users/gregorlauter/tools/anaconda3/envs/pormake/lib/python3.9/site-packages/pormake/locator.py:20: UserWarning: Optimal rotation is not uniquely or poorly defined for the given sets of vectors.
  U, rmsd = scipy.spatial.transform.Rotation.align_vectors(p, q)
>>> Start finding bonds in generated